# 02 — Ensembles

## Objetivo

Como Bagging e Boosting se comportam no mesmo problema?

Random Forest representa Bagging; Gradient Boosting é o boosting principal;
XGBoost aparece uma única vez como comparação avançada, sem tuning.

In [2]:
from pathlib import Path
import sys

ponto_atual = Path.cwd().resolve()
RAIZ = next(
    caminho for caminho in (ponto_atual, *ponto_atual.parents)
    if (caminho / "data" / "raw" / "UCI_Credit_Card.csv").exists()
)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


import time
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import average_precision_score
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

from src.auxiliares import COLUNAS_NOMINAIS
from src.visual_utils import grafico_comparacao_modelos

pasta_processados = RAIZ / "data" / "processed"
dados_treino = pd.read_csv(pasta_processados / "treino.csv")
dados_validacao = pd.read_csv(pasta_processados / "validacao.csv")

X_treino = dados_treino.drop(columns=["inadimplente"])
y_treino = dados_treino["inadimplente"]

X_validacao = dados_validacao.drop(columns=["inadimplente"])
y_validacao = dados_validacao["inadimplente"]

## 2.1 — Bagging: qual é a referência?

In [6]:
preprocessamento_arvores = ColumnTransformer(
    [(
        "nominais",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        COLUNAS_NOMINAIS,
    )],
    remainder="passthrough",
    verbose_feature_names_out=False
)

modelo_random_forest = Pipeline([
    ("preprocessamento", preprocessamento_arvores),
    ("modelo" , RandomForestClassifier(n_estimators=200, 
                                       random_state=42,
                                       n_jobs=-1))
])

inicio = time.perf_counter()
modelo_random_forest.fit(X_treino, y_treino)
tempo_random_forest = time.perf_counter() - inicio

previsoes_random_forest = modelo_random_forest.predict(X_validacao)
probabilidades_random_forest = modelo_random_forest.predict_proba(X_validacao)[:,1]

ap_random_forest = average_precision_score(
    y_validacao,
    probabilidades_random_forest,
)

resultados = pd.DataFrame([

    {
    "modelo": "Random Forest",
    "average_precision": ap_random_forest,
    "tempo_treino_s":tempo_random_forest,
    "Precision": precision_score(y_validacao, previsoes_random_forest),
    "Recall": recall_score(y_validacao, previsoes_random_forest),
    "F1": f1_score(y_validacao, previsoes_random_forest),        
    }
            ])

resultados

,modelo,average_precision,tempo_treino_s,Precision,Recall,F1
0,Random Forest,0.525546,2.304559,0.657609,0.357724,0.46338


## 2.2 — Boosting: o Gradient Boosting avança?

In [8]:
preprocessamento_arvores = ColumnTransformer(
    [(
        "nominais",
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        COLUNAS_NOMINAIS,
    )],
    remainder="passthrough",
    verbose_feature_names_out=False
)

modelo_gradient_boosting = Pipeline([
    ("preprocessamento", preprocessamento_arvores),
    ("modelo" , GradientBoostingClassifier(random_state=42,))
])

inicio = time.perf_counter()
modelo_gradient_boosting.fit(X_treino, y_treino)
tempo_gradient_boosting = time.perf_counter() - inicio

previsoes_gradient_boosting = modelo_gradient_boosting.predict(X_validacao)
probabilidades_gradient_boosting = modelo_gradient_boosting.predict_proba(X_validacao)[:,1]

ap_gradient_boosting = average_precision_score(
    y_validacao,
    probabilidades_gradient_boosting,
)

resultados = pd.DataFrame([

    {
    "modelo": "Grandient Boosting",
    "average_precision": ap_gradient_boosting,
    "tempo_treino_s":tempo_gradient_boosting,
    "Precision": precision_score(y_validacao, previsoes_gradient_boosting),
    "Recall": recall_score(y_validacao, previsoes_gradient_boosting),
    "F1": f1_score(y_validacao, previsoes_gradient_boosting),        
    }
            ])

resultados

,modelo,average_precision,tempo_treino_s,Precision,Recall,F1
0,Grandient Boosting,0.551675,13.489177,0.687964,0.342203,0.457058


## 2.3 — Quanto o XGBoost acrescenta sem tuning?

## 2.4 — Qual ensemble se sai melhor na validação?

In [ ]:
fig = grafico_comparacao_modelos(resultados, "Bagging e Boosting na validação")
fig.show()

## 2.5 — Resultado

O Gradient Boosting fica muito próximo do XGBoost e mantém a implementação
principal dentro do scikit-learn. O ganho pequeno do XGBoost não muda o foco do
curso.